In [1]:
import argparse
import time
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
# Filepaths
stop_times_path = 'Datasets/stop_times.txt'
stops_path = 'Datasets/stops.txt'
trips_path = 'Datasets/trips.txt'
calendar_path = 'Datasets/calendar.txt'
api_path = 'Datasets/api_mar1.csv'

In [3]:
def process_gtfs_time(time_str):
    '''
    This function gets the time string from GTFS dataset and turns it into minutes since midnight
    '''

    try:
        h, m, s = map(int, str(time_str).split(':'))
        return h*60 + m + s/60
    except Exception:
        return np.nan
    
def process_api_time(df):
    dt = pd.to_datetime(df)
    return dt.dt.hour * 60 + dt.dt.minute + dt.dt.second/60

In [4]:
def preprocess_static_data(stop_times_path, stops_path, trips_path, calendar_path, cache_dir='gtfs_cache'):
    '''
    This function merges stop_times, trips, stops, and their calendars in a Parquet file
    *** OFFLINE STEP
    '''

    print('Pre-processing GTFS data')

    # Setting the directories
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    out_path = cache_dir / 'static_data.parquet'
    cal_path = cache_dir / 'trips_calendar.parquet'

    # Initializing clock for performance analysis
    t0 = time.time()

    # Reading trips and stop files
    trips = pd.read_csv(trips_path, dtype={
        'route_id': str,
        'trip_id': str,
        'shape_id': str,
        'service_id': str
        })
    
    stops = pd.read_csv(stops_path, dtype={
        'stop_id': str
    })

    # Building the merged dataset (using chunks because of large size)
    trip_id_set = set(trips['trip_id'].astype(str))
    chunks = []
    for chunk in pd.read_csv(stop_times_path, 
                             dtype={'trip_id': str, 'stop_id': str},
                             chunksize=500_000):
        chunks.append(chunk[chunk['trip_id'].isin(trip_id_set)])
    
    stop_times = pd.concat(chunks, ignore_index=True)

    # Adjusting data types
    stop_times['arrival_minutes'] = stop_times['arrival_time'].apply(process_gtfs_time)
    stop_times['departure_minutes'] = stop_times['departure_time'].apply(process_gtfs_time)
    stop_times['shape_dist_traveled'] = pd.to_numeric(stop_times['shape_dist_traveled'], errors='coerce')
    stop_times = stop_times.dropna(subset=["shape_dist_traveled", "arrival_minutes"])
    stop_times = stop_times.sort_values(["trip_id", "stop_sequence"]).reset_index(drop=True)

    # Merging datasets
    stop_times = stop_times.merge(
        stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']],
        on = 'stop_id', how = 'left'
    )

    stop_times = stop_times.merge(
        trips[['trip_id', 'route_id', 'shape_id', 'direction', 'schd_trip_id', 'service_id']],
        on='trip_id', how='left')
    
    stop_times['route_id'] = stop_times['route_id'].astype(str).str.strip()
    important_cols = ['route_id', 'trip_id', 'shape_id', 'stop_sequence', 'arrival_minutes', 'departure_minutes',
                      'shape_dist_traveled', 'stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'service_id', 
                      'direction', 'schd_trip_id']
    
    stop_times = stop_times[[c for c in important_cols if c in stop_times.columns]]
    stop_times = stop_times.sort_values(['route_id', 'shape_id', 'trip_id', 'stop_sequence'])

    # Building the calendar lookup for trips
    calendar = pd.read_csv(calendar_path, dtype=str)
    day_cols = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']
    for col in day_cols:
        calendar[col] = calendar[col].astype(int)
    trips_calendar = trips[['route_id', 'service_id', 'trip_id', 'shape_id', 'schd_trip_id']].merge(
        calendar[['service_id'] + day_cols], on = 'service_id', how = 'left'
    )

    # Writing Parquet tables
    pq.write_table(
        pa.Table.from_pandas(stop_times, preserve_index=False),
        out_path, row_group_size=100_000, compression='snappy'
    )
    
    pq.write_table(
        pa.Table.from_pandas(trips_calendar, preserve_index=False),
        cal_path, compression='snappy'
        )
    
    print('GTFS pre-processed')
    

In [5]:
def load_gtfs_cache(cache_dir='gtfs_cache'):
    '''
    This function reads the pre-processed gtfs parquet files
    '''
    print('Loading GTFS pre-processed parquet files')

    cache_dir = Path(cache_dir)
    out_path = cache_dir / 'static_data.parquet'
    cal_path = cache_dir / 'trips_calendar.parquet'

    if not out_path.exists():
        raise FileNotFoundError('GTFS cache not found — rerun the pre-processing')
    
    t0 = time.time()
    con = duckdb.connect()
    stop_times = con.execute(f"SELECT * FROM read_parquet('{out_path}')").df()
    trips_calendar = con.execute(f"SELECT * FROM read_parquet('{cal_path}')").df()

    con.close()

    print('GTFS loaded')

    return stop_times, trips_calendar

In [6]:
def build_trip_index(stop_times, trips_calendar):
    '''
    Converts GTFS stop_times into fast lookup structures for matching.

    Returns:
        trip_store:  {trip_id: dict}          — full trip data as numpy arrays
        shape_index: {shape_id: [trip_id, …]} — all trips per shape, for fallback matching
    '''
    print('Building trip index...')
    t0 = time.time()

    # Build service_id → active weekday set {0=Mon … 6=Sun}
    day_cols = ['monday','tuesday','wednesday','thursday','friday','saturday','sunday']
    svc_days = {}
    trip_svc = {}
    if all(c in trips_calendar.columns for c in day_cols):
        for _, row in trips_calendar.iterrows():
            sid    = str(row['service_id'])
            active = {i for i, col in enumerate(day_cols) if int(row[col]) == 1}
            svc_days[sid] = active
        trip_svc = dict(zip(
            trips_calendar['trip_id'].astype(str),
            trips_calendar['service_id'].astype(str)
        ))

    trip_store  = {}  # trip_id → arrays
    shape_index = {}  # shape_id → [trip_id, ...]

    for (route_id, trip_id, shape_id), grp in stop_times.groupby(['route_id', 'trip_id', 'shape_id']):
        route_id = str(route_id).strip()
        trip_id  = str(trip_id)
        shape_id = str(shape_id)
        grp      = grp.sort_values('stop_sequence')
        svc_id   = trip_svc.get(trip_id)

        trip_store[trip_id] = {
            'trip_id':         trip_id,
            'route_id':        route_id,
            'shape_id':        shape_id,
            'service_id':      svc_id,
            'active_days':     svc_days.get(svc_id, set(range(7))),
            'dists':           grp['shape_dist_traveled'].values.astype(np.float32),
            'arr_min':         grp['arrival_minutes'].values.astype(np.float32),
            'dep_min':         grp['departure_minutes'].values.astype(np.float32),
            'stop_ids':        grp['stop_id'].values,
            'stop_names':      grp['stop_name'].values,
            'stop_lats':       grp['stop_lat'].values.astype(np.float32),
            'stop_lons':       grp['stop_lon'].values.astype(np.float32),
            # First-stop anchor
            'first_stop_name': grp['stop_name'].values[0],
            'first_dep_min':   float(grp['departure_minutes'].values[0]),
            'first_dep_sec':   int(round(float(grp['departure_minutes'].values[0]) * 60)),
        }

        if shape_id not in shape_index:
            shape_index[shape_id] = []
        shape_index[shape_id].append(trip_id)

    print(f'Trip index built: {len(trip_store):,} trips | {len(shape_index):,} shapes | {time.time()-t0:.1f}s')
    return trip_store, shape_index

In [7]:
def build_stst_index(trip_store, shape_index, unique_pids):
    '''
    Builds (route_id, pid_str, dep_seconds) → trip_id lookup.

    route_id is included because the same pid suffix can appear on multiple
    routes, and two routes can share a first-stop departure time.
    dep_seconds matches the raw stst column in the vehicle feed (int, seconds
    past midnight) so the lookup is a direct dict get() with no conversion.

    Called after load_api_data() so actual vehicle pid values are known.
    '''
    print('Building stst_index...')
    t0 = time.time()

    stst_index = {}
    for pid_str in unique_pids:
        pid_str = str(pid_str).strip()
        if pid_str in ('', 'nan', 'None'):
            continue
        for shape_id, trip_ids in shape_index.items():
            if str(shape_id).endswith(pid_str):
                for trip_id in trip_ids:
                    trip = trip_store[trip_id]
                    key  = (trip['route_id'], pid_str, trip['first_dep_sec'])
                    stst_index[key] = trip_id

    print(f'stst_index built: {len(stst_index):,} keys | {len(unique_pids):,} unique pids | {time.time()-t0:.1f}s')
    return stst_index

In [8]:
def load_api_data(api_path):
    '''This function loads the API data using DuckDB'''

    print('Loading API data')
    t0 = time.time()

    path = Path(api_path)
    chunks = []
    for chunk in pd.read_csv(path, 
                             dtype=str,
                             chunksize=500_000):
        chunks.append(chunk)
    
    df = pd.concat(chunks, ignore_index=True)

    # Adjusting data types
    df['vid'] = df['vid'].astype(str).str.strip()
    df['tmstmp'] = pd.to_datetime(df['tmstmp'], errors='coerce')
    df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
    df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
    df['pid'] = df['pid'].astype(str).str.strip()
    df['dly'] = df['dly'].astype(str).str.lower().map({'true':True, 'false':False}).fillna(False)
    df['rt'] = df['rt'].str.strip()
    df['pdist'] = pd.to_numeric(df['pdist'], errors='coerce')
    df['tatripid'] = pd.to_numeric(df['tatripid'], errors='coerce')
    df['origtatripno'] = pd.to_numeric(df['origtatripno'], errors='coerce')
    df['stst'] = pd.to_numeric(df['stst'], errors='coerce')
    df['time_minutes'] = process_api_time(df['tmstmp'])

    df = df.rename(columns={
        'vid': 'vehicle_id',
        'rt': 'route_id'})

    df.head()

    orig = len(df)
    # Dropping rows with no timestamp or location
    df = df.dropna(subset=['tmstmp', 'pdist', 'lat', 'lon']).reset_index(drop=True)
    
    before = len(df)

    before = len(df)
    df = df[df['time_minutes'] > 0].reset_index(drop=True)
    dropped = before - len(df)
    if dropped:
        print(f'Dropped {dropped:,} rows with malformed timestamps (time_minutes=0)')

    # Dropping duplicate observations
    df = df.drop_duplicates(subset=['tmstmp', 'vehicle_id'], keep='first').reset_index(drop=True)

    final = len(df)
    
    print(f'Original dataset length: {orig}\n')
    print(f'Null timestamp or location values: {before-orig}; filtered length: {before}\n')
    print(f'Duplicate observations: {before-final}; filtered length: {final}\n')

    return df

In [9]:
def _get_trip_date(tmstmp, stst_sec, rollback_hour=4):
    '''
    Returns the calendar date a trip belongs to.

    If the observation is in the early morning (before rollback_hour)
    and stst is in the late evening (>= 20:00), the trip started the
    previous calendar day — roll back the date by one day.
    '''
    obs_date = tmstmp.date()
    if tmstmp.hour < rollback_hour and stst_sec >= 20 * 3600:
        obs_date = (tmstmp - pd.Timedelta(days=1)).date()
    return obs_date


def _slim_dtypes(df):
    '''
    Reduce memory footprint before writing to parquet:
      - low-cardinality string columns → Categorical (dictionary-encoded in parquet)
      - float64 coordinates/distances  → float32
      - small integer columns          → int16/int32
    '''
    cat_cols = ['vehicle_id', 'route_id', 'pid', 'dly', 'tatripid', 'origtatripno']
    f32_cols = ['lat', 'lon', 'pdist', 'time_minutes', 'stst']
    i16_cols = ['weekday']
    i32_cols = ['run_id']

    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in f32_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')
    for col in i16_cols:
        if col in df.columns:
            df[col] = df[col].astype('int16')
    for col in i32_cols:
        if col in df.columns:
            df[col] = df[col].astype('int32')
    return df


def preprocess_api_data(api_path, cache_dir='api_cache', gap_minutes=60):
    '''
    Cleans API CSV, assigns trip_date / run_id, optimizes dtypes, and writes Parquet.
    *** OFFLINE STEP — run once per raw CSV file.
    '''
    print('Pre-processing API data...')
    t0 = time.time()

    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    out_path = cache_dir / 'api_data.parquet'

    df = load_api_data(api_path)

    # Compute trip_date, overnight flag, weekday
    df['trip_date'] = [_get_trip_date(ts, st) for ts, st in zip(df['tmstmp'], df['stst'])]
    df['_overnight'] = df['trip_date'] != df['tmstmp'].dt.date
    df['weekday']    = df['tmstmp'].dt.weekday

    # Vectorized run_id assignment
    group_cols = ['vehicle_id', 'route_id', 'pid', 'stst', 'trip_date']
    df = df.sort_values(group_cols + ['tmstmp']).reset_index(drop=True)

    gc           = df[group_cols].fillna('__NA__')
    group_change = (gc != gc.shift()).any(axis=1)
    time_gap     = df['time_minutes'].diff()
    time_gap[group_change] = 0
    df['run_id'] = (group_change | (time_gap > gap_minutes)).cumsum()

    # Run size — single-observation runs get stst lookup skipped at match time
    run_cols = group_cols + ['run_id']
    df['run_size'] = df.groupby(run_cols)['vehicle_id'].transform('size').astype(np.int16)

    # Slim dtypes — reduces parquet size and reload memory significantly
    df = _slim_dtypes(df)
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f'In-memory size after optimization: {mem_mb:.0f} MB')

    pq.write_table(
        pa.Table.from_pandas(df, preserve_index=False),
        out_path, row_group_size=100_000, compression='snappy'
    )
    print(f'API data pre-processed: {len(df):,} rows | {time.time()-t0:.1f}s')

In [10]:
def load_api_cache(cache_dir='api_cache'):
    '''
    Reads the pre-processed API Parquet file.
    '''
    print('Loading API cache...')
    t0 = time.time()

    out_path = Path(cache_dir) / 'api_data.parquet'
    if not out_path.exists():
        raise FileNotFoundError('API cache not found — run preprocess_api_data() first')

    con = duckdb.connect()
    df  = con.execute(f"SELECT * FROM read_parquet('{out_path}')").df()
    con.close()

    df['tmstmp']    = pd.to_datetime(df['tmstmp'])
    df['trip_date'] = pd.to_datetime(df['trip_date']).dt.date

    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f'API cache loaded: {len(df):,} rows | {mem_mb:.0f} MB | {time.time()-t0:.1f}s')
    return df

# MAIN CODE

In [11]:
stop_times, trips_calendar = load_gtfs_cache()

# Run once to build the cache, then comment out:
# preprocess_api_data(api_path)

Loading GTFS pre-processed parquet files
GTFS loaded


In [12]:
api = load_api_cache()

Loading API cache...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

API cache loaded: 4,969,622 rows | 3831 MB | 9.6s


In [13]:
trip_store, shape_index = build_trip_index(stop_times, trips_calendar)
unique_pids = api['pid'].dropna().unique()
stst_index  = build_stst_index(trip_store, shape_index, unique_pids)

# Lookup at match time:
# key = (row['route_id'], str(row['pid']), int(row['stst']))
# trip_id = stst_index.get(key)  → exact trip or None

Building trip index...
Trip index built: 49,936 trips | 782 shapes | 7.0s
Building stst_index...
stst_index built: 45,407 keys | 756 unique pids | 0.1s


In [14]:
# ── helpers used by match_trips_bus ──────────────────────────────────────────

def _find_bracket(dists, pdist):
    b = int(np.searchsorted(dists, pdist, side='right') - 1)
    if b < 0 or b >= len(dists) - 1:
        return None
    return b


def _interpolate_scheduled_time(trip, b, pdist):
    dists      = trip['dists']
    dist_range = float(dists[b + 1] - dists[b])
    if dist_range <= 0:
        return float(trip['dep_min'][b])
    frac = (pdist - float(dists[b])) / dist_range
    return float(trip['dep_min'][b]) + frac * (float(trip['arr_min'][b + 1]) - float(trip['dep_min'][b]))


def _compute_delay(row, trip):
    b = _find_bracket(trip['dists'], row['pdist'])

    if b is None:
        return {
            'delay_minutes':             None,
            'schedule_elapsed_minutes':  None,
            'first_stop_name':           trip['first_stop_name'],
            'scheduled_departure_stop1': trip['first_dep_min'],
            'prev_stop_name':            None,
            'next_stop_name':            None,
        }

    interp_time = _interpolate_scheduled_time(trip, b, row['pdist'])

    obs_time = row['time_minutes']
    if row.get('_overnight', False):
        obs_time += 1440

    schedule_elapsed = interp_time - trip['first_dep_min']
    delay            = obs_time - interp_time

    return {
        'delay_minutes':             round(delay, 1),
        'schedule_elapsed_minutes':  round(schedule_elapsed, 1),
        'first_stop_name':           trip['first_stop_name'],
        'scheduled_departure_stop1': trip['first_dep_min'],
        'prev_stop_name':            trip['stop_names'][b],
        'next_stop_name':            trip['stop_names'][b + 1],
    }


def _score_candidates(candidates, trip_store, pdist, obs_time, max_score_minutes=45):
    best_trip_id = None
    best_score   = np.inf

    for trip_id in candidates:
        trip = trip_store[trip_id]
        b    = _find_bracket(trip['dists'], pdist)
        if b is None:
            continue
        score = abs(obs_time - _interpolate_scheduled_time(trip, b, pdist))
        if score < best_score:
            best_score   = score
            best_trip_id = trip_id

    return best_trip_id if best_score <= max_score_minutes else None


def match_trips_bus(api, trip_store, shape_index, stst_index):
    print('Matching vehicles to trips...')
    t0 = time.time()

    route_index = {}
    for trip_id, trip in trip_store.items():
        route_index.setdefault(trip['route_id'], []).append(trip_id)

    results    = []
    group_cols = ['vehicle_id', 'route_id', 'pid', 'stst', 'trip_date', 'run_id']

    for group_keys, grp in api.groupby(group_cols, dropna=False):
        vehicle_id, route_id, pid_str, stst_sec, trip_date, run_id = group_keys
        weekday     = int(grp['weekday'].iloc[0])
        pid_str     = str(pid_str).strip()
        stst_int    = int(stst_sec) if pd.notna(stst_sec) else None
        is_overnight= grp['_overnight'].any()
        run_size    = int(grp['run_size'].iloc[0])

        obs_time_median = float(grp['time_minutes'].median())
        if is_overnight:
            obs_time_median += 1440

        # ── 1. stst exact lookup ──────────────────────────────────────────
        trip_id    = None
        match_type = 'unscheduled'

        if stst_int is not None and run_size > 1:
            candidate = stst_index.get((route_id, pid_str, stst_int))
            if candidate:
                trip = trip_store[candidate]
                if abs(obs_time_median - trip['first_dep_min']) <= 50:
                    trip_id    = candidate
                    match_type = 'stst'
                else:
                    match_type = 'unscheduled'  # rejected — fall through to scoring

        # ── 2. shape fallback (pid-based) ─────────────────────────────────
        if trip_id is None:
            pid_candidates = [
                t for shape_id, trip_ids in shape_index.items()
                if str(shape_id).endswith(pid_str)
                for t in trip_ids
                if trip_store[t]['route_id'] == route_id
                and weekday in trip_store[t]['active_days']
            ]

            if pid_candidates:
                trip_id = _score_candidates(pid_candidates, trip_store,
                                            float(grp['pdist'].median()),
                                            obs_time_median)
                if trip_id:
                    match_type = 'fallback_pid'

        # ── 3. full route fallback ────────────────────────────────────────
        if trip_id is None:
            route_candidates = [
                t for t in route_index.get(route_id, [])
                if weekday in trip_store[t]['active_days']
            ]
            if route_candidates:
                trip_id = _score_candidates(route_candidates, trip_store,
                                            float(grp['pdist'].median()),
                                            obs_time_median)
                if trip_id:
                    match_type = 'fallback_route'

        # ── 4. compute delay per observation ──────────────────────────────
        trip = trip_store.get(trip_id) if trip_id else None
        for idx, row in grp.iterrows():
            result = _compute_delay(row, trip) if trip else {
                'delay_minutes':             None,
                'schedule_elapsed_minutes':  None,
                'first_stop_name':           None,
                'scheduled_departure_stop1': None,
                'prev_stop_name':            None,
                'next_stop_name':            None,
            }
            results.append({**row, **result,
                            'matched_trip_id': trip_id,
                            'match_type':      match_type})

    result_df = pd.DataFrame(results).reset_index(drop=True)

    n_total        = len(result_df)
    n_stst         = (result_df['match_type'] == 'stst').sum()
    n_fb_pid       = (result_df['match_type'] == 'fallback_pid').sum()
    n_fb_route     = (result_df['match_type'] == 'fallback_route').sum()
    n_unscheduled  = (result_df['match_type'] == 'unscheduled').sum()
    print(f'Total: {n_total:,} | '
          f'stst: {n_stst:,} ({n_stst/n_total*100:.1f}%) | '
          f'fallback_pid: {n_fb_pid:,} ({n_fb_pid/n_total*100:.1f}%) | '
          f'fallback_route: {n_fb_route:,} ({n_fb_route/n_total*100:.1f}%) | '
          f'unscheduled: {n_unscheduled:,} ({n_unscheduled/n_total*100:.1f}%)')
    print(f'Done in {time.time()-t0:.1f}s')
    
    result_df['pre_trip'] = (result_df['pdist'] == 0) & (result_df['delay_minutes'] < -15)

    return result_df

In [15]:
result = match_trips_bus(api, trip_store, shape_index, stst_index)

Matching vehicles to trips...
Total: 4,969,622 | stst: 4,116,117 (82.8%) | fallback_pid: 733,494 (14.8%) | fallback_route: 13,280 (0.3%) | unscheduled: 106,731 (2.1%)
Done in 280.6s


In [26]:
# def identify_ghost_buses(result, trip_store, trips_calendar,
#                          shapes_path='Datasets/shapes.txt',
#                          freeze_minutes=20, jump_threshold=10000,
#                          offroute_meters=2000):
def identify_ghost_buses(result, trip_store, trips_calendar,
                         shapes_path='Datasets/shapes.txt',
                         freeze_minutes=20, jump_threshold=10000,
                         offroute_meters=2000,
                         min_obs_for_g4=400_000):
    '''
    Identifies ghost bus signals across four dimensions:

    G1 — Freeze:          pdist unchanged for >= freeze_minutes within a run
    G2 — Jump:            pdist jumps > jump_threshold feet between consecutive
                          observations within a run
    G3 — Off-route:       GPS coordinates > offroute_meters from matched shape
    G4 — Never departed:  scheduled bus trip with active service on an observed
                          date has zero non-pre_trip matched observations —
                          appended as synthetic rows with match_type='ghost_trip'

    Returns result DataFrame with ghost_freeze, ghost_jump, ghost_offroute,
    is_ghost columns added, and G4 ghost trips appended as rows.
    '''
    print('Identifying ghost buses...')
    t0 = time.time()

    # Strip any existing G4 rows — makes function idempotent
    df = result[result['match_type'] != 'ghost_trip'].copy()

    eligible = df['matched_trip_id'].notna() & ~df['pre_trip'].fillna(False)

    df['ghost_freeze']   = False
    df['ghost_jump']     = False
    df['ghost_offroute'] = False

    matched = df[eligible].copy()
    matched = matched.sort_values(['vehicle_id', 'run_id', 'tmstmp'])
    grp_cols = ['vehicle_id', 'run_id']

    # ── G1: Freeze ────────────────────────────────────────────────────────
    print('  G1: detecting freezes...')
    pdist_changed = matched.groupby(grp_cols)['pdist'].transform(
        lambda x: x != x.shift()
    )
    matched['_last_change_time'] = matched['time_minutes'].where(pdist_changed)
    matched['_last_change_time'] = matched.groupby(grp_cols)['_last_change_time'].transform('ffill')
    frozen_duration = matched['time_minutes'] - matched['_last_change_time']
    df.loc[matched.index[frozen_duration >= freeze_minutes], 'ghost_freeze'] = True
    matched = matched.drop(columns=['_last_change_time'])

    # ── G2: Jump ──────────────────────────────────────────────────────────
    print('  G2: detecting jumps...')
    pdist_diff = matched.groupby(grp_cols)['pdist'].diff().abs()
    df.loc[matched.index[pdist_diff > jump_threshold], 'ghost_jump'] = True

    # ── G3: Off-route ─────────────────────────────────────────────────────
    print('  G3: detecting off-route observations...')
    shapes = pd.read_csv(shapes_path, dtype={'shape_id': str})
    shapes = shapes.sort_values(['shape_id', 'shape_pt_sequence'])

    shape_coords = {}
    for shape_id, grp in shapes.groupby('shape_id'):
        shape_coords[str(shape_id)] = (
            grp['shape_pt_lat'].values.astype(np.float64),
            grp['shape_pt_lon'].values.astype(np.float64)
        )

    trip_to_shape = {tid: t['shape_id'] for tid, t in trip_store.items()}
    matched['_shape_id'] = matched['matched_trip_id'].map(trip_to_shape)

    R = 6_371_000.0
    offroute_index = []
    for shape_id, grp in matched.groupby('_shape_id'):
        if shape_id not in shape_coords:
            continue
        s_lats, s_lons = shape_coords[shape_id]
        obs_lats = grp['lat'].values
        obs_lons = grp['lon'].values

        dlat = np.radians(s_lats[None, :] - obs_lats[:, None])
        dlon = np.radians(s_lons[None, :] - obs_lons[:, None])
        a    = (np.sin(dlat / 2) ** 2
                + np.cos(np.radians(obs_lats))[:, None]
                * np.cos(np.radians(s_lats))[None, :]
                * np.sin(dlon / 2) ** 2)
        min_dist = R * 2 * np.arcsin(np.sqrt(a.clip(0, 1)).min(axis=1))
        offroute_mask = min_dist > offroute_meters
        offroute_index.extend(grp.index[offroute_mask].tolist())

    df.loc[offroute_index, 'ghost_offroute'] = True
    matched = matched.drop(columns=['_shape_id'])

    # ── G4: Never departed ────────────────────────────────────────────────
    print('  G4: detecting never-departed trips...')

    # Only include dates with sufficient observations to avoid
    # false ghost positives from incomplete data collection windows
    obs_counts = df[df['match_type'] != 'ghost_trip'].groupby('trip_date').size()
    obs_dates  = set(obs_counts[obs_counts >= min_obs_for_g4].index)
    print(f'  G4: using {len(obs_dates)} complete dates '
          f'(excluded {(obs_counts < min_obs_for_g4).sum()} incomplete)')
    bus_routes     = set(df['route_id'].dropna().unique())  # all routes in vehicle data
    departed_trips = set(df.loc[eligible, 'matched_trip_id'].dropna().unique())

    day_cols = ['monday','tuesday','wednesday','thursday','friday','saturday','sunday']
    svc_days = {}
    for _, row in trips_calendar.iterrows():
        sid    = str(row['service_id'])
        active = {i for i, col in enumerate(day_cols) if int(row[col]) == 1}
        svc_days[sid] = active

    ghost_rows = []
    for trip_id, trip in trip_store.items():
        if trip['route_id'] not in bus_routes:  # skip rail
            continue
        if trip_id in departed_trips:
            continue
        active_days = trip['active_days']
        for obs_date in sorted(obs_dates):
            if obs_date.weekday() in active_days:
                ghost_rows.append({
                    'matched_trip_id':           trip_id,
                    'route_id':                  trip['route_id'],
                    'shape_id':                  trip['shape_id'],
                    'first_stop_name':           trip['first_stop_name'],
                    'scheduled_departure_stop1': trip['first_dep_min'],
                    'trip_date':                 obs_date,
                    'match_type':                'ghost_trip',
                    'ghost_freeze':              False,
                    'ghost_jump':                False,
                    'ghost_offroute':            False,
                    'is_ghost':                  True,
                    'pre_trip':                  False,
                    'delay_minutes':             None,
                })
                break

    ghost_df = pd.DataFrame(ghost_rows).reindex(columns=df.columns)
    for col in ['ghost_freeze', 'ghost_jump', 'ghost_offroute', 'is_ghost', 'pre_trip']:
        ghost_df[col] = False
    ghost_df['is_ghost'] = True

    ghost_df = ghost_df.fillna(np.nan)
    df = pd.concat([df, ghost_df], ignore_index=True)

    for col in ['ghost_freeze', 'ghost_jump', 'ghost_offroute', 'is_ghost', 'pre_trip']:
        df[col] = df[col].astype(bool)

    df['is_ghost'] = (df['ghost_freeze'] | df['ghost_jump'] |
                      df['ghost_offroute'] | (df['match_type'] == 'ghost_trip'))

    # ── Summary ───────────────────────────────────────────────────────────
    n_eligible = eligible.sum()
    n_freeze   = df['ghost_freeze'].sum()
    n_jump     = df['ghost_jump'].sum()
    n_offroute = df['ghost_offroute'].sum()
    n_g123     = (df['ghost_freeze'] | df['ghost_jump'] | df['ghost_offroute']).sum()
    n_g4       = len(ghost_rows)

    print(f'  G1 freeze:       {n_freeze:>7,} obs  ({n_freeze/n_eligible*100:.2f}%)')
    print(f'  G2 jump:         {n_jump:>7,} obs  ({n_jump/n_eligible*100:.2f}%)')
    print(f'  G3 off-route:    {n_offroute:>7,} obs  ({n_offroute/n_eligible*100:.2f}%)')
    print(f'  G1/G2/G3 total:  {n_g123:>7,} obs  ({n_g123/n_eligible*100:.2f}%)')
    print(f'  G4 never-depart: {n_g4:>7,} trips appended')
    print(f'  Done in {time.time()-t0:.1f}s')

    return df

In [27]:
result = identify_ghost_buses(
    result, trip_store, trips_calendar,
    shapes_path='Datasets/shapes.txt'
)

Identifying ghost buses...
  G1: detecting freezes...
  G2: detecting jumps...
  G3: detecting off-route observations...
  G4: detecting never-departed trips...
  G4: using 7 complete dates (excluded 3 incomplete)


/var/folders/y1/1zf0l7g530dd_mqbfll_qkhh0000gn/T/ipykernel_11350/4134067637.py:143: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ghost_df = ghost_df.fillna(np.nan)
/var/folders/y1/1zf0l7g530dd_mqbfll_qkhh0000gn/T/ipykernel_11350/4134067637.py:144: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, ghost_df], ignore_index=True)


  G1 freeze:        11,509 obs  (0.24%)
  G2 jump:           3,359 obs  (0.07%)
  G3 off-route:        470 obs  (0.01%)
  G1/G2/G3 total:   15,308 obs  (0.32%)
  G4 never-depart:   2,359 trips appended
  Done in 177.6s


In [35]:
def analyze_route_performance(result, trip_store,
                               late_threshold=3, very_late_threshold=15,
                               early_threshold=-1):
    '''
    Computes per-route performance metrics from matched vehicle observations.

    Metrics (overall + per time-of-day bucket):
        otp_pct        — % observations on time (early_threshold <= delay <= late_threshold)
        early_pct      — % observations early (delay < early_threshold)
        late_pct       — % observations late (delay > late_threshold)
        very_late_pct  — % observations very late (delay > very_late_threshold)
        delay_mean     — mean delay across clean observations (minutes)
        delay_median   — median delay
        delay_std      — standard deviation of delay (reliability proxy)
        delay_p90      — 90th percentile delay
        n_obs          — total clean observations used
        n_trips        — unique matched trips
        ghost_rate_pct — G4 never-departed as % of (matched + ghost) trips
        n_ghost_trips  — count of G4 never-departed trips

    Time-of-day buckets (based on time_minutes):
        early_morning  — 00:00–05:59  (owl / first runs)
        am_peak        — 06:00–09:59
        midday         — 10:00–14:59
        pm_peak        — 15:00–18:59
        evening        — 19:00–23:59
    '''

    # ── Clean observations only ───────────────────────────────────────────
    clean_mask = (
        result['match_type'].isin(['stst', 'fallback_pid', 'fallback_route']) &
        ~result['pre_trip'].fillna(False) &
        ~result['is_ghost'].fillna(False) &
        result['delay_minutes'].notna()
    )
    clean = result[clean_mask].copy()

    # ── Time-of-day bucket ────────────────────────────────────────────────
    bins   = [-1, 360, 600, 900, 1140, 1441]
    labels = ['early_morning', 'am_peak', 'midday', 'pm_peak', 'evening']
    clean['tod'] = pd.cut(clean['time_minutes'], bins=bins, labels=labels, right=False)

    # ── Classify each observation ─────────────────────────────────────────
    clean['_early']     = clean['delay_minutes'] < early_threshold
    clean['_on_time']   = clean['delay_minutes'].between(early_threshold, late_threshold)
    clean['_late']      = clean['delay_minutes'] > late_threshold
    clean['_very_late'] = clean['delay_minutes'] > very_late_threshold

    def _agg_metrics(df):
        n = len(df)
        if n == 0:
            return pd.Series({
                'n_obs':          0,
                'delay_mean':     np.nan,
                'delay_median':   np.nan,
                'delay_std':      np.nan,
                'delay_p90':      np.nan,
                'early_pct':      np.nan,
                'on_time_pct':    np.nan,
                'otp_pct':        np.nan,
                'late_pct':       np.nan,
                'very_late_pct':  np.nan,
            })
        return pd.Series({
            'n_obs':          n,
            'delay_mean':     df['delay_minutes'].mean(),
            'delay_median':   df['delay_minutes'].median(),
            'delay_std':      df['delay_minutes'].std(),
            'delay_p90':      df['delay_minutes'].quantile(0.90),
            'early_pct':      df['_early'].mean()     * 100,
            'on_time_pct':    df['_on_time'].mean()   * 100,
            'otp_pct':        (df['_early'] | df['_on_time']).mean() * 100,
            'late_pct':       df['_late'].mean()      * 100,
            'very_late_pct':  df['_very_late'].mean() * 100,
        })

    # ── Overall metrics ───────────────────────────────────────────────────
    overall = (clean.groupby('route_id')
                    .apply(_agg_metrics, include_groups=False)
                    .reset_index())
    overall.columns = ['route_id'] + [f'overall_{c}' for c in overall.columns[1:]]

    # unique trips — computed separately
    n_trips = (clean.groupby('route_id')['matched_trip_id']
                    .nunique()
                    .rename('overall_n_trips')
                    .reset_index())
    overall = overall.merge(n_trips, on='route_id', how='left')

    # ── Per time-of-day metrics ───────────────────────────────────────────
    tod_agg = (clean.groupby(['route_id', 'tod'], observed=True)
                    .apply(_agg_metrics, include_groups=False)
                    .reset_index())

    # Pivot so each tod bucket becomes a column prefix
    tod_pivot = tod_agg.pivot(index='route_id', columns='tod')
    tod_pivot.columns = [f'{tod}_{metric}' for metric, tod in tod_pivot.columns]
    tod_pivot = tod_pivot.reset_index()

    # ── Ghost rate ────────────────────────────────────────────────────────
    ghost = (result[result['match_type'] == 'ghost_trip']
             .groupby('route_id')
             .size()
             .rename('n_ghost_trips')
             .reset_index())

    # ── Combine ───────────────────────────────────────────────────────────
    perf = (overall
            .merge(tod_pivot,  on='route_id', how='left')
            .merge(ghost,      on='route_id', how='left'))

    perf['n_ghost_trips']  = perf['n_ghost_trips'].fillna(0).astype(int)
    perf['ghost_rate_pct'] = (perf['n_ghost_trips'] /
                              (perf['overall_n_trips'] + perf['n_ghost_trips']) * 100)

    # Round all float columns
    float_cols = perf.select_dtypes(include='float').columns
    perf[float_cols] = perf[float_cols].round(2)

    perf = perf.sort_values('overall_otp_pct').reset_index(drop=True)

    print(f'Route performance computed: {len(perf)} routes | '
          f'{clean_mask.sum():,} clean observations')

    print(f'\n=== WORST 10 ROUTES BY OTP ===')
    cols = ['route_id', 'overall_otp_pct', 'overall_late_pct',
            'overall_very_late_pct', 'overall_delay_std', 'ghost_rate_pct']
    print(perf[cols].head(10).to_string())

    print(f'\n=== PEAK VS OFF-PEAK DELAY (worst 10) ===')
    peak_cols = ['route_id',
                 'am_peak_otp_pct', 'am_peak_delay_median',
                 'pm_peak_otp_pct', 'pm_peak_delay_median',
                 'midday_otp_pct',  'midday_delay_median']
    print(perf[peak_cols].head(10).to_string())

    return perf

In [36]:
perf = analyze_route_performance(result, trip_store)

Route performance computed: 123 routes | 4,772,676 clean observations

=== WORST 10 ROUTES BY OTP ===
  route_id  overall_otp_pct  overall_late_pct  overall_very_late_pct  overall_delay_std  ghost_rate_pct
0      169            45.57             54.43                   5.39               5.92           18.18
1       62            51.52             48.48                   8.46               7.90           10.25
2      201            55.53             44.47                   2.48               5.06            2.16
3      171            58.99             41.01                  20.18              12.14           16.80
4       50            60.61             39.39                   3.10               5.61            4.42
5      206            61.80             38.20                   3.53               5.09            0.00
6      54A            65.00             35.00                   3.18               5.57            0.00
7       65            65.66             34.34                   4.

In [37]:
clean_mask = (
    result['match_type'].isin(['stst', 'fallback_pid', 'fallback_route']) &
    ~result['pre_trip'].fillna(False) &
    ~result['is_ghost'].fillna(False) &
    result['delay_minutes'].notna()
)
clean = result[clean_mask].copy()

In [39]:
result.columns

Index(['vehicle_id', 'tmstmp', 'lat', 'lon', 'hdg', 'pid', 'route_id', 'des',
       'pdist', 'dly', 'tatripid', 'origtatripno', 'tablockid', 'zone', 'mode',
       'psgld', 'stst', 'stsd', 'pulled_at', 'rt_chunk', 'time_minutes',
       'trip_date', '_overnight', 'weekday', 'run_id', 'run_size',
       'delay_minutes', 'schedule_elapsed_minutes', 'first_stop_name',
       'scheduled_departure_stop1', 'prev_stop_name', 'next_stop_name',
       'matched_trip_id', 'match_type', 'pre_trip', 'ghost_freeze',
       'ghost_jump', 'ghost_offroute', 'is_ghost'],
      dtype='object')

In [42]:
# 1. Route 169 deep dive — what's driving the midday collapse?
r169 = clean[(clean['route_id'] == '169')]
print(r169.groupby('prev_stop_name')['delay_minutes']
      .agg(['mean','count'])
      .sort_values('mean', ascending=False)
      .head(15).to_string())

# 2. PM peak ranking across all routes
print(perf[['route_id','pm_peak_otp_pct','pm_peak_delay_median',
            'am_peak_otp_pct','am_peak_delay_median']]
      .sort_values('pm_peak_otp_pct')
      .head(15).to_string())

                                    mean  count
prev_stop_name                                 
Ups Facility Stop 1             7.719355     31
79th Street & Pulaski           6.129897     97
79th Street & Cicero            5.924752    303
Ups Facility Stop 2             5.341250    320
Halsted & 79th Street           4.480672    119
79th Street & Kedzie            4.079817    109
69th Street & Wentworth         4.070370     54
Halsted & 69th Street           3.798824     85
79th Street & Ashland           3.724138    116
79th Street & Western           3.548148    108
69th Street & State (Red Line)  2.866667     69
   route_id  pm_peak_otp_pct  pm_peak_delay_median  am_peak_otp_pct  am_peak_delay_median
3       171            29.47                  9.70            87.98                  -0.4
0       169            40.65                  4.70            45.48                   3.9
2       201            47.48                  3.35            64.72                   1.4
8       54B     